# New Requirements per Cycle Table
Parses the selection history file and counts how many **new** requirement IDs appear in each cycle compared to the previous one.

In [ ]:
# ======================================================
# Cell 1: Environment Setup — Run this first!
# ======================================================
# Detects Google Colab vs local Jupyter and sets OUTPUT_DIR,
# which is the Hist-RTSGA folder where History_Tables/ lives.
#
# ── Google Colab users ──────────────────────────────────────────────
#   Upload the project to Google Drive, then set DRIVE_PROJECT_PATH
#   below to the folder that contains the Hist-RTSGA/ subfolder.
#   Run the Hist-RTSGA notebook first so History_Tables/ exists.
#
# ── Local Jupyter users ─────────────────────────────────────────────
#   No action needed. OUTPUT_DIR is set to the notebook's folder.
# ======================================================

import subprocess, sys, os
from pathlib import Path

for _pkg in ["pandas", "openpyxl", "matplotlib"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", _pkg, "-q"])
print("✅ Dependencies ready.")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Set DRIVE_PROJECT_PATH to the Drive folder that contains Hist-RTSGA/
    DRIVE_PROJECT_PATH = "/content/drive/MyDrive/BV-Driven-History-RTSGA"  # ← adjust if needed

    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUTPUT_DIR = str(Path(DRIVE_PROJECT_PATH) / "Hist-RTSGA")
        _hist_tables = Path(OUTPUT_DIR) / "History_Tables"
        if not _hist_tables.exists():
            print(f"⚠️  History_Tables/ not found at {_hist_tables}.")
            print("   Run the Hist-RTSGA notebook first to generate it.")
    except Exception as _e:
        print(f"Drive mount failed ({_e}). Set OUTPUT_DIR manually below.")
        OUTPUT_DIR = "/content"
else:
    OUTPUT_DIR = str(Path.cwd())

print(f"\nEnvironment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Output dir  : {OUTPUT_DIR}")

In [ ]:
# ======================================================
# Cell 2: Configuration
# ======================================================
import os

# ── Input: selection history Excel file ───────────────────────────────────
# Built from OUTPUT_DIR (set by Cell 1) — matches where Hist-RTSGA saves it.
HISTORY_FILE = os.path.join(OUTPUT_DIR, "History_Tables", "selection_history_ALL_COMBINED.xlsx")

if not os.path.exists(HISTORY_FILE):
    raise FileNotFoundError(
        f"File not found at:\n  {HISTORY_FILE}\n"
        "Run the Hist-RTSGA notebook first (Cell 5: Selection History Tables), "
        "or update OUTPUT_DIR in Cell 1 to point to the correct folder."
    )

# ── Configurations to include ─────────────────────────────────────────────
# Set to None to use ALL configurations found in the file.
SELECTED_CONFIGS = [
    'Tol0.9_Wt0.05', 'Tol0.9_Wt0.1', 'Tol0.9_Wt0.15', 'Tol0.9_Wt0.2', 'Tol0.9_Wt0.25',
    'Tol0.8_Wt0.05', 'Tol0.8_Wt0.1', 'Tol0.8_Wt0.15', 'Tol0.8_Wt0.2', 'Tol0.8_Wt0.25',
    'Tol0.7_Wt0.05', 'Tol0.7_Wt0.1', 'Tol0.7_Wt0.15', 'Tol0.7_Wt0.2', 'Tol0.7_Wt0.25',
]

TABLE_TITLE = "Number of New Requirements per Cycle (vs. Previous Cycle)"

# ── Output ────────────────────────────────────────────────────────────────
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "new_requirements_per_cycle.xlsx")

print(f"📄 History file : {HISTORY_FILE}")
print(f"   Configurations: {SELECTED_CONFIGS if SELECTED_CONFIGS else 'ALL'}")
print(f"   Output file   : {OUTPUT_FILE}")

In [6]:
# ======================================================
# Cell 3: Parse & Compute New Requirements per Cycle
# ======================================================
import pandas as pd
import re

def parse_req_ids(cell_str):
    """Extract requirement IDs from a string like '(3,89), (6,13), ...'"""
    if pd.isna(cell_str):
        return set()
    return set(int(m) for m in re.findall(r'\((\d+),', str(cell_str)))

# Load the file
df_raw = pd.read_excel(HISTORY_FILE)
cycle_cols = [c for c in df_raw.columns if re.match(r'^C\d+$', c)]
num_cycles = len(cycle_cols)
print(f"✅ Loaded {len(df_raw)} configurations, {num_cycles} cycles.")
print(f"   All configs: {df_raw['Configuration'].tolist()}")

# Filter to selected configurations
if SELECTED_CONFIGS:
    missing = [c for c in SELECTED_CONFIGS if c not in df_raw['Configuration'].values]
    if missing:
        raise ValueError(f"These configurations were not found in the file: {missing}")
    df_raw = df_raw[df_raw['Configuration'].isin(SELECTED_CONFIGS)].copy()
    # Preserve the order specified in SELECTED_CONFIGS
    df_raw['_order'] = df_raw['Configuration'].map({c: i for i, c in enumerate(SELECTED_CONFIGS)})
    df_raw = df_raw.sort_values('_order').drop(columns='_order').reset_index(drop=True)

print(f"\n📊 Computing new requirements per cycle for {len(df_raw)} configuration(s)...")

results = []   # list of dicts: {Configuration, C1, C2, ..., C20}

for _, row in df_raw.iterrows():
    config_name = row['Configuration']
    entry = {'Configuration': config_name}
    prev_reqs = set()

    for col in cycle_cols:
        cur_reqs = parse_req_ids(row[col])
        # C1: no prior cycle, so all selected reqs are "new"
        new_reqs = cur_reqs if col == 'C1' else cur_reqs - prev_reqs
        entry[col] = len(new_reqs)
        prev_reqs = cur_reqs

    results.append(entry)

df_new_reqs = pd.DataFrame(results)
print("\n✅ Done. Preview (first 5 cycles):")
print(df_new_reqs[['Configuration'] + cycle_cols[:5]].to_string(index=False))


✅ Loaded 15 configurations, 20 cycles.
   All configs: ['Tol0.9_Wt0.05', 'Tol0.9_Wt0.1', 'Tol0.9_Wt0.15', 'Tol0.9_Wt0.2', 'Tol0.9_Wt0.25', 'Tol0.8_Wt0.05', 'Tol0.8_Wt0.1', 'Tol0.8_Wt0.15', 'Tol0.8_Wt0.2', 'Tol0.8_Wt0.25', 'Tol0.7_Wt0.05', 'Tol0.7_Wt0.1', 'Tol0.7_Wt0.15', 'Tol0.7_Wt0.2', 'Tol0.7_Wt0.25']

📊 Computing new requirements per cycle for 15 configuration(s)...

✅ Done. Preview (first 5 cycles):
Configuration  C1  C2  C3  C4  C5
Tol0.9_Wt0.05   7   0   0   0   0
 Tol0.9_Wt0.1   7   0   0   0   0
Tol0.9_Wt0.15   7   0   0   0   0
 Tol0.9_Wt0.2   7   0   0   0   0
Tol0.9_Wt0.25   7   0   0   0   0
Tol0.8_Wt0.05   7   0   0   0   0
 Tol0.8_Wt0.1   7   0   0   0   0
Tol0.8_Wt0.15   7   0   0   0   0
 Tol0.8_Wt0.2   7   0   0   0   0
Tol0.8_Wt0.25   7   0   0   0   0
Tol0.7_Wt0.05   7   0   0   0   0
 Tol0.7_Wt0.1   7   0   0   0   0
Tol0.7_Wt0.15   7   0   0   0   0
 Tol0.7_Wt0.2   7   0   0   0   0
Tol0.7_Wt0.25   7   0   0   0   0


In [7]:
# ======================================================
# Cell 4: Display Split Table (C1–C10 | C11–C20)
# ======================================================
from IPython.display import display, HTML
import math

all_cycles  = [c for c in df_new_reqs.columns if re.match(r'^C\d+$', c)]
half        = math.ceil(len(all_cycles) / 2)
first_half  = all_cycles[:half]          # C1–C10
second_half = all_cycles[half:]          # C11–C20

left_df  = df_new_reqs[['Configuration'] + first_half].copy()
right_df = df_new_reqs[['Configuration'] + second_half].copy()

# Rename 'Configuration' to 'Cycle' style header for display (matches the image)
left_df  = left_df.rename(columns={'Configuration': 'Config'})
right_df = right_df.rename(columns={'Configuration': 'Config'})

# ── Render as a side-by-side HTML table ──────────────────────────────────
def df_to_html_rows(df):
    rows = ""
    for _, row in df.iterrows():
        cells = "".join(f"<td style='text-align:center; padding:4px 10px;'>{v}</td>"
                        for v in row)
        rows += f"<tr>{cells}</tr>"
    return rows

def make_header(cols, label='Cycle'):
    cells = f"<th style='padding:4px 10px; border-bottom:2px solid black;'>{label}</th>"
    for c in cols[1:]:
        cells += f"<th style='padding:4px 10px; border-bottom:2px solid black; text-align:center;'>{c}</th>"
    return f"<tr>{cells}</tr>"

th_style = "font-family: serif; font-size: 14px; text-align: center; margin-bottom: 6px;"

left_header  = make_header(left_df.columns,  label='Config')
right_header = make_header(right_df.columns, label='Config')

html = f"""
<div style="font-family: serif;">
  <p style="{th_style}"><b>{TABLE_TITLE}</b></p>
  <table style="border-collapse: collapse; display: inline-block; vertical-align: top; margin-right: 30px;">
    <thead style="border-top: 2px solid black;">{left_header}</thead>
    <tbody style="border-bottom: 2px solid black;">{df_to_html_rows(left_df)}</tbody>
  </table>
  <table style="border-collapse: collapse; display: inline-block; vertical-align: top;">
    <thead style="border-top: 2px solid black;">{right_header}</thead>
    <tbody style="border-bottom: 2px solid black;">{df_to_html_rows(right_df)}</tbody>
  </table>
</div>
"""
display(HTML(html))


Config,C1,C2,C3,C4,C5,C6,C7,C8,C9,C10
Tol0.9_Wt0.05,7,0,0,0,0,0,0,0,0,0
Tol0.9_Wt0.1,7,0,0,0,0,0,0,0,0,0
Tol0.9_Wt0.15,7,0,0,0,0,0,0,0,0,0
Tol0.9_Wt0.2,7,0,0,0,0,0,0,0,0,0
Tol0.9_Wt0.25,7,0,0,0,0,0,0,0,0,0
Tol0.8_Wt0.05,7,0,0,0,0,0,0,0,0,0
Tol0.8_Wt0.1,7,0,0,0,0,0,0,0,0,0
Tol0.8_Wt0.15,7,0,0,0,0,0,0,0,0,0
Tol0.8_Wt0.2,7,0,0,0,0,0,0,0,0,0
Tol0.8_Wt0.25,7,0,0,0,0,0,0,0,0,0


In [13]:
# ======================================================
# Cell 5: Save to Excel (optional)
# ======================================================
if OUTPUT_FILE:
    with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
        # Full table on one sheet
        df_new_reqs.to_excel(writer, sheet_name='New_Reqs_Per_Cycle', index=False)

        # Split view matching the display above
        left_save  = df_new_reqs[['Configuration'] + first_half]
        right_save = df_new_reqs[['Configuration'] + second_half]

        left_save.to_excel(writer,  sheet_name='C1_to_C10',  index=False)
        right_save.to_excel(writer, sheet_name='C11_to_C20', index=False)

    print(f"✅ Saved to: {OUTPUT_FILE}")
    print(f"   Sheets: New_Reqs_Per_Cycle | C1_to_C10 | C11_to_C20")
else:
    print("OUTPUT_FILE is None — skipping save.")


✅ Saved to: /Users/aumgarasia/Desktop/BV-Driven-History-RTSGA/Hist-RTSGA/new_requirements_per_cycle.xlsx
   Sheets: New_Reqs_Per_Cycle | C1_to_C10 | C11_to_C20
